In [ ]:
import os
from os.path import expanduser
home = expanduser("~/")

import sys
sys.path.insert(0, home+'/gigalens'+'/src')
print('MASTER BRANCH GIGALENS')

srcdir = os.path.join(home, "gigalens/src/")

In [ ]:
from gigalens.jax.inference import ModellingSequence
from gigalens.jax.model import ForwardProbModel, BackwardProbModel
from gigalens.model import PhysicalModel
from gigalens.jax.simulator import LensSimulator
from gigalens.simulator import SimulatorConfig
from gigalens.jax.profiles.light import sersic
from gigalens.jax.profiles.mass import epl, shear

import tensorflow_probability.substrates.jax as tfp
import jax
from jax import random
import numpy as np
import optax
from jax import numpy as jnp
from matplotlib import pyplot as plt
import optax
from helpers import *
tfd = tfp.distributions

In [ ]:
prior = make_default_prior()
kernel = np.load(os.path.join(srcdir, 'gigalens/assets/psf.npy')).astype(np.float32)
sim_config = SimulatorConfig(delta_pix=0.065, num_pix=80, supersample=2, kernel=kernel)
phys_model = PhysicalModel([epl.EPL(50), shear.Shear()], [sersic.SersicEllipse(use_lstsq=False)], [sersic.SersicEllipse(use_lstsq=False)])
lens_sim = LensSimulator(phys_model, sim_config, bs=1)

systems_dir = os.path.join(home, "GIGALens-Code/SystemSaves")
f = np.load(os.path.join(systems_dir, "100SystemsStandard80px.npz"))
observed_imgs = jnp.array([f[key] for key in f.files])

observed_img = observed_imgs[4]

prob_model = ForwardProbModel(prior, observed_img, background_rms=0.2, exp_time=100)
model_seq = ModellingSequence(phys_model, prob_model, sim_config)

In [ ]:
map_optimizer = optax.adabelief(1e-2, b1=0.95, b2=0.99) #nesterov=True may not be implemented in current optax
map_estimate, map_loss_hist = model_seq.MAP(map_optimizer, seed=0, n_samples=5000, num_steps=1000)

In [ ]:
plt.plot(map_loss_hist)
plt.ylim(bottom=0, top=3)
plt.show()

In [ ]:
lps = prob_model.log_prob(LensSimulator(phys_model, sim_config, bs=5000), map_estimate)[0]
best = map_estimate[jnp.nanargmax(lps)][jnp.newaxis,:]

In [ ]:
svi_optimizer = optax.adabelief(1e-4, b1=0.95, b2=0.99)
qz, loss_hist = model_seq.SVI(best, svi_optimizer, n_vi=1000, num_steps=1500)

In [ ]:
plt.plot(loss_hist)

In [ ]:
samples = model_seq.HMC(qz, num_burnin_steps=250, num_results=750)

In [ ]:
smp = jnp.transpose(samples.all_states, (1, 2, 0, 3)).reshape((-1, 22))
hmc_median = jnp.median(smp, axis=0)[jnp.newaxis,:]

In [ ]:
rhat= tfp.mcmc.potential_scale_reduction(jnp.transpose(samples.all_states, (1,2,0,3)), independent_chain_ndims=2)

In [ ]:
lens_sim_prob = LensSimulator(phys_model, sim_config, bs=1)
def log_prob(params):
    lps = prob_model.log_prob(lens_sim_prob, params)[0]
    return lps

map_hessian = jnp.squeeze(jax.hessian(log_prob)(best))
map_hess_cov = -jnp.linalg.inv(map_hessian)
map_hess_qz = tfp.distributions.MultivariateNormalFullCovariance(loc=jnp.squeeze(best), covariance_matrix=map_hess_cov)

hmc_hessian = jnp.squeeze(jax.hessian(log_prob)(hmc_median))
hmc_hess_cov = -jnp.linalg.inv(hmc_hessian)
hmc_hess_qz = tfp.distributions.MultivariateNormalFullCovariance(loc=jnp.squeeze(hmc_median), covariance_matrix=hmc_hess_cov)

In [ ]:
cov_diff = (map_hess_qz.covariance() - qz.covariance())/qz.covariance()
print(f"Mean error: {100*jnp.mean(cov_diff)}%, Max Error: {100*jnp.max(cov_diff)}%")

In [ ]:
tups = [(0, 0), (0, 1), (1, 0), (2, 0)]
ind = np.flip(np.argsort(lps))
map_samples_sorted = map_estimate[ind]
top_10 = map_samples_sorted[:1000]
top_10_x = prob_model.bij.forward(list(top_10.T))
top_10_plot = np.vstack([np.array(list(top_10_x[i][j].values())) for i, j in tups]).T

In [ ]:
#* Thought is, in unconstrained space

# cov = qz.covariance()
# mean = qz.loc

mean = np.mean(smp, axis=0)
cov = np.cov(smp.T)

qz_hmc = tfp.distributions.MultivariateNormalFullCovariance(loc=mean, covariance_matrix=cov.astype(np.float32))
dist = map_estimate-mean
y = np.sqrt(np.einsum('ki,ij,kj->k', dist, np.linalg.inv(cov), dist))#(map_estimate-hmc_median) @ (np.linalg.inv(scale @ scale.T) @ (map_estimate-hmc_median).T)

In [ ]:
map_5sigma = map_estimate[y < 6]
map_5sigma_x = prob_model.bij.forward(list(map_5sigma.T))
map_5sigma_plot = np.vstack([np.array(list(map_5sigma_x[i][j].values())) for i, j in tups]).T

In [ ]:
elbo_lens_sim = LensSimulator(phys_model, sim_config, bs=1000)
def elbo(qz_calc):
    z = qz_calc.sample(1000, seed=jax.random.PRNGKey(0))
    lps = qz_calc.log_prob(z)
    model_log_prob = prob_model.log_prob(elbo_lens_sim, z)[0]

    return jnp.mean(lps - model_log_prob)

print(f"Hess_qz ELBO: {elbo(map_hess_qz)}")
print(f"HMC qz fit ELBO {elbo(qz_hmc)}")
print(f"SVI ELBO: {elbo(qz)}")

In [ ]:
map_samples_x = prob_model.bij.forward(list(map_estimate.T))
map_best_x = prob_model.bij.forward(list(best.T))

svi_samples_z = qz.sample(1000, seed=jax.random.PRNGKey(0))
svi_samples_x = prob_model.bij.forward(list(svi_samples_z.T))

hess_samples_z = map_hess_qz.sample(1000, seed=jax.random.PRNGKey(0))
hess_samples_x = prob_model.bij.forward(list(hess_samples_z.T))

hmc_samples_x = prob_model.bij.forward(list(smp.T))
labels = cornerplot_labels(map_best_x)

fig = cornerplot_posterior(labels, svi_samples_x, overplots=map_best_x, color='blue', overplot_color='red')
cornerplot_posterior(labels, hmc_samples_x, fig=fig)
cornerplot_posterior(labels, hess_samples_x, fig=fig, color='red')
# cornerplot_posterior(labels, map_5sigma_x, fig=fig, color='orange')
# for sample in map_5sigma_plot:
#     corner.overplot_points(fig, sample[np.newaxis], marker="*", color="C1")
plt.show()

In [ ]:
rhat

In [ ]:
print(map_samples_x[0][0]['theta_E'].shape)